# Diagnóstico de Telemetría y Detección de Anomalías — UNAM Motorsports DAQ

Reclutamiento DAQ, Fórmula SAE. Pipeline reproducible sobre `data.csv` (100 Hz).

**Estado:** borrador de trabajo. Varios umbrales están marcados como `PROVISIONAL`
en `pipeline.py` y dependen de las respuestas de la sesión de dudas del 14 de
julio (ver `docs/PREGUNTAS_MANANA.md`). La arquitectura del pipeline (filtro,
cálculo de G, diagrama G-G, detector con regla de persistencia de 100ms) ya
es funcional y no cambia; lo que se recalibra son los umbrales exactos.


In [ ]:
import sys
sys.path.insert(0, ".")
from pipeline import *
import matplotlib.pyplot as plt

df = load_telemetry("../data/data.csv")
df = add_filtered_channels(df)
df.head()


## 1. Acondicionamiento de señal

Filtro Butterworth pasa-bajas de **fase cero** (`scipy.signal.filtfilt`, orden
4, corte 10 Hz) sobre `accel_x` y `accel_y`. Fase cero = no introduce retraso
temporal, requisito explícito de la actividad.

**Por qué el valor crudo es engañoso:** la señal cruda mezcla la aceleración
real del vehículo con ruido de alta frecuencia (vibración estructural,
resonancia del chasis, ruido eléctrico del ADC). Si se calcula |G| máxima
sobre la señal cruda, un solo pico de ruido puede inflar artificialmente el
valor reportado — por eso la magnitud debe calcularse **después** de filtrar.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(df["Timestamp"], df["accel_x"], alpha=0.4, label="accel_x cruda")
axes[0].plot(df["Timestamp"], df["accel_x_filt"], label="accel_x filtrada", linewidth=1.5)
axes[0].set_ylabel("accel_x [g]")
axes[0].legend()

axes[1].plot(df["Timestamp"], df["accel_y"], alpha=0.4, label="accel_y cruda")
axes[1].plot(df["Timestamp"], df["accel_y_filt"], label="accel_y filtrada", linewidth=1.5)
axes[1].set_ylabel("accel_y [g]")
axes[1].set_xlabel("Tiempo [s]")
axes[1].legend()
plt.tight_layout()
plt.savefig("../web-dashboard/fig_acondicionamiento.png", dpi=100)
plt.show()


In [ ]:
print(f"|G| máxima sobre señal FILTRADA: {max_g_combined(df):.3f} g")
print(f"|G| máxima sobre señal CRUDA (referencia, no usar para reportar): {df['g_combined_raw'].max():.3f} g")


## 2. Diagrama G-G

Envolvente de fricción del auto: cada punto es (accel_x_filtrada,
accel_y_filtrada) en un instante. Un diagrama simétrico indica que el auto
frena/acelera y gira igual de fuerte en ambas direcciones; una asimetría
indica un problema físico de set-up (ej. reparto de frenada desbalanceado,
IMU no alineada con el eje del auto) o saturación del sensor en un eje.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(df["accel_x_filt"], df["accel_y_filt"], s=2, alpha=0.3)
ax.set_xlabel("accel_x filtrada [g]")
ax.set_ylabel("accel_y filtrada [g]")
ax.set_title("Diagrama G-G")
ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig("../web-dashboard/fig_gg_diagram.png", dpi=100)
plt.show()


**Lectura del diagrama (PROVISIONAL, a completar tras confirmar el dataset
final):** revisar si la envolvente es más ancha hacia un lado del eje X o Y.
Si `accel_x` está saturada con más frecuencia en un signo que en otro, eso
apunta a que el sensor se satura en maniobras de un solo tipo (ej. solo en
frenadas fuertes, no en aceleración), lo cual es un problema de rango de
calibración del sensor, no de la física del auto.


## 3. Detección de fallas de integridad del sensor inercial (pérdida de señal)

Se busca un tramo donde `accel_x`/`accel_y` dejen de variar de forma
realista (varianza colapsada en una ventana móvil), como si el sensor se
hubiera "congelado" en vez de seguir midiendo ruido normal de vibración.

**PROVISIONAL:** con el dataset actual no se encontró ningún segmento de
este tipo. Ver pregunta 1 de `docs/PREGUNTAS_MANANA.md` — falta confirmar
cómo está codificada la pérdida de señal en el archivo definitivo.


In [ ]:
dropout_segments = detect_signal_dropout(df)
print(f"Segmentos candidatos a pérdida de señal (>100ms): {len(dropout_segments)}")
for s in dropout_segments:
    print(f"  t={s.start_t:.2f}s -> t={s.end_t:.2f}s ({s.duration_ms:.0f} ms)")


## 4. Violaciones de plausibilidad (freno + acelerador simultáneos)

Regla de persistencia: solo cuenta como falla si `brake_pressure` y
`throttle` superan el umbral **al mismo tiempo, de forma continua, por más
de 100 ms** (evita falsos positivos de un solo sample).

**PROVISIONAL:** umbral de 50% en ambos canales (ver pregunta 3 de
`docs/PREGUNTAS_MANANA.md`). Con este umbral aparece **un solo evento**
sostenido en todo el registro, consistente con que el PDF lo describe como
un evento raro y puntual a reportar con timestamp exacto — buena señal de
que el umbral va en la dirección correcta, pendiente de confirmar el valor
exacto mañana.


In [ ]:
plaus_segments = detect_plausibility_violations(df)
print(f"Violaciones de plausibilidad (>100ms, umbral {BRAKE_THROTTLE_THRESHOLD_PCT}%): {len(plaus_segments)}")
for s in plaus_segments:
    print(f"  EVENTO: t={s.start_t:.2f}s -> t={s.end_t:.2f}s (duración {s.duration_ms:.0f} ms, "
          f"muestras [{s.start_idx}:{s.end_idx}])")


## 5. Saturación del sensor inercial

Se marca saturación cuando `|accel_x|` o `|accel_y|` superan 0.98 (cerca del
límite físico del rango del sensor), sostenido por más de 100 ms.


In [ ]:
sat_segments = detect_sensor_saturation(df)
print(f"Segmentos de saturación (>100ms): {len(sat_segments)}")
for s in sat_segments:
    print(f"  EVENTO: t={s.start_t:.2f}s -> t={s.end_t:.2f}s (duración {s.duration_ms:.0f} ms)")


## 6. Reporte técnico (borrador, máx. 300 palabras)

*Pendiente de redactar la versión final una vez confirmados los umbrales
exactos en la sesión de dudas del 14 de julio. Borrador de estructura:*

- **Acondicionamiento:** filtro Butterworth orden 4, corte 10 Hz, fase cero.
  |G| máxima filtrada vs. cruda, y por qué se reporta la filtrada.
- **Diagrama G-G:** descripción de la asimetría observada y su causa física
  probable.
- **Integridad del sensor:** resultado de la búsqueda de pérdida de señal
  (pendiente confirmar patrón esperado).
- **Plausibilidad:** timestamp exacto del evento detectado, duración, y
  justificación del umbral usado.
- **Acciones recomendadas:** ej. revisar calibración del sensor en el eje
  saturado, y auditar el log del piloto en el timestamp del evento de
  plausibilidad antes de usar este tramo de datos para decisiones de
  ingeniería.
